# 🚁 Disaster Detection Drone — Training Pipeline

Fine-tunes **YOLOv11n** on three pinned Roboflow aerial-disaster datasets (flood, landslide, earthquake)
and visualizes the confusion matrix, training curves, and validation predictions for each model.

**This is the cleaned-up version of the original research notebook:**
- The Roboflow API key is read from the `ROBOFLOW_API_KEY` environment variable — never hardcoded.
  On Kaggle, attach it via *Add-ons → Secrets* (or set `os.environ` below before running).
- Dataset versions are pinned for reproducibility.
- Run output paths are taken from Ultralytics' `results.save_dir` (the original notebook guessed a
  wrong path and crashed on the visualization cells).
- One loop trains all three models instead of three copy-pasted blocks.

Companion package: [`drone_detect/`](../drone_detect/) in the repo root provides the same pipeline as a CLI.


In [ ]:
# Core dependencies (pre-installed on Kaggle/Colab images, pinned here for reproducibility)
%pip install -q "ultralytics>=8.3.0" "roboflow>=1.1.0" "pyyaml>=6.0"


In [ ]:
import os
from pathlib import Path

from roboflow import Roboflow
from ultralytics import YOLO
from IPython.display import Image, display

# --- Configuration -----------------------------------------------------
# Kaggle: attach ROBOFLOW_API_KEY via Add-ons -> Secrets and enable it for this notebook.
API_KEY = os.environ["ROBOFLOW_API_KEY"]  # raises a clear KeyError if the secret is missing
WORKSPACE = "ashwins-workspace-zz9gf"

# Mirrors configs/datasets.yaml in the repo root. Bump `version` when Roboflow publishes a new one.
DATASETS = {
    "flood": {
        "project": "ai-disaster-survival-drone",
        "version": 2,
        "run_name": "flood_model",
        "description": "Flood extent and survivor detection (Hurricane Harvey aerial imagery)",
    },
    "landslide": {
        "project": "ai-disaster-drone-landslide",
        "version": 1,
        "run_name": "landslide_model",
        "description": "Landslide and slope-failure detection from UAV imagery",
    },
    "earthquake": {
        "project": "ai-disaster-drone-earthquakes",
        "version": 1,
        "run_name": "earthquake_model",
        "description": "Earthquake rubble and structural damage assessment",
    },
}

TRAIN_ARGS = dict(
    epochs=50,
    imgsz=640,
    batch=-1,       # auto-batch: largest batch that fits GPU memory
    patience=20,    # early stopping when validation mAP plateaus
    cos_lr=True,    # cosine LR schedule
    seed=42,        # reproducibility
    plots=True,
)


In [ ]:
# --- Download + train all three models ---------------------------------

run_dirs = {}  # disaster name -> Ultralytics save_dir for the training run

rf = Roboflow(api_key=API_KEY)
for name, cfg in DATASETS.items():
    print(f"\n{'=' * 70}\n  {name.upper()} — {cfg['description']}\n{'=' * 70}")

    # Download (reuses the local copy if the version was already fetched)
    dataset = (
        rf.workspace(WORKSPACE)
        .project(cfg["project"])
        .version(cfg["version"])
        .download(model_format="yolov11", location=f"datasets/{cfg['project']}-v{cfg['version']}")
    )

    # Train: COCO-pretrained nano -> disaster specialist
    model = YOLO("yolo11n.pt")
    results = model.train(
        data=str(Path(dataset.location) / "data.yaml"),
        project="runs/drone_models",
        name=cfg["run_name"],
        **TRAIN_ARGS,
    )
    run_dirs[name] = Path(results.save_dir)
    print(f"best weights: {run_dirs[name] / 'weights' / 'best.pt'}")


In [ ]:
# --- Visualize each training run ---------------------------------------
# save_dir comes straight from Ultralytics, so these paths always exist —
# the original notebook hard-coded a wrong runs/ path here.

for name, run in run_dirs.items():
    print(f"\n{'=' * 70}\n  {name.upper()} — results in {run}\n{'=' * 70}")
    print("-- Confusion matrix --")
    display(Image(filename=str(run / "confusion_matrix.png"), width=800))
    print("-- Training curves --")
    display(Image(filename=str(run / "results.png"), width=800))
    print("-- Validation predictions --")
    display(Image(filename=str(run / "val_batch0_pred.jpg"), width=800))


In [ ]:
# --- Sanity check: inference on a validation image --------------------

weights = run_dirs["flood"] / "weights" / "best.pt"
val_images = sorted((run_dirs["flood"].parent.parent / "datasets").rglob("*valid*"))
sample = None
for d in val_images:  # first directory that actually contains images
    imgs = [p for p in d.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    if imgs:
        sample = imgs[0]
        break

if sample is None:
    print("No validation image found — skipping the sanity check.")
else:
    model = YOLO(str(weights))
    res = model.predict(source=str(sample), conf=0.25, save=True)[0]
    print(f"{len(res.boxes)} detection(s) on {sample.name}:")
    for box in res.boxes:
        cls = model.names[int(box.cls)]
        print(f"  {cls:<20} conf={float(box.conf):.2f}")
    display(Image(filename=res.path, width=800))


## Next steps

- Export for onboard inference: `python -m drone_detect export --weights runs/drone_models/flood_model/weights/best.pt --format onnx`
  (or `--format engine` for Jetson TensorRT).
- Pull the repo's `drone_detect` CLI for scripted training outside notebooks.
- See the repo README for the full architecture and roadmap.
